# RepGeo Visualization

Supports two generation-result schemas:
- With `sublayer_metrics`: plot block/attention/MLP `para_perp_ratio` summaries.
- With only `layer_metrics`: plot block-level summaries first; attention/MLP cannot be recovered from that file.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
from pathlib import Path


In [ ]:
SAVE_FIG = False
OUT_DIR = Path('../results')

data = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
runs = data['runs'] if 'runs' in data else [data]
print(f'loaded runs: {len(runs)}')

def pick_steps(run):
    timeline = run.get('timeline') or []
    if timeline:
        dec = [s for s in timeline if s.get('phase') == 'decode']
        if dec:
            return dec
        return timeline
    return run.get('steps', [])


def _stats_from_grouped(by_key):
    keys = sorted(by_key.keys())
    vals = [by_key[k] for k in keys]
    means = [sum(v) / len(v) for v in vals]
    mins = [min(v) for v in vals]
    maxs = [max(v) for v in vals]
    return keys, means, mins, maxs


def collect_layerwise_from_layer_metrics(steps, value_key):
    by_layer = {}
    for step in steps:
        for m in (step.get('layer_metrics') or []):
            layer = m.get('layer')
            v = m.get(value_key)
            if layer is None or v is None:
                continue
            by_layer.setdefault(int(layer), []).append(float(v))
    if not by_layer:
        return [], [], [], []
    return _stats_from_grouped(by_layer)


def collect_layerwise_from_sublayer(steps, metric_key, value_key):
    by_layer = {}
    for step in steps:
        for item in (step.get('sublayer_metrics') or []):
            layer = item.get('layer')
            v = (item.get(metric_key) or {}).get(value_key)
            if layer is None or v is None:
                continue
            by_layer.setdefault(int(layer), []).append(float(v))
    if not by_layer:
        return [], [], [], []
    return _stats_from_grouped(by_layer)


def collect_stepwise_from_layer_metrics(steps, value_key):
    by_step = {}
    for s in steps:
        st = s.get('step')
        if st is None:
            continue
        values = [m.get(value_key) for m in (s.get('layer_metrics') or []) if m.get(value_key) is not None]
        if values:
            by_step.setdefault(int(st), []).extend(float(v) for v in values)
    if not by_step:
        return [], [], [], []
    return _stats_from_grouped(by_step)


def collect_stepwise_from_sublayer(steps, metric_key, value_key):
    by_step = {}
    for s in steps:
        st = s.get('step')
        if st is None:
            continue
        values = []
        for item in (s.get('sublayer_metrics') or []):
            v = (item.get(metric_key) or {}).get(value_key)
            if v is not None:
                values.append(float(v))
        if values:
            by_step.setdefault(int(st), []).extend(values)
    if not by_step:
        return [], [], [], []
    return _stats_from_grouped(by_step)


series = [('block', 'BLOCK', 'tab:blue'), ('attn', 'ATTN', 'tab:orange'), ('mlp', 'MLP', 'tab:green')]
key_map = {'block': 'block_update', 'attn': 'attn_update', 'mlp': 'mlp_update'}

for prompt_idx, run in enumerate(runs):
    steps = pick_steps(run)
    if not steps:
        print(f'prompt {prompt_idx}: no steps, skipped')
        continue

    has_sublayer = any(('sublayer_metrics' in s and s['sublayer_metrics']) for s in steps)
    has_layer = any(('layer_metrics' in s and s['layer_metrics']) for s in steps)
    print(f'prompt {prompt_idx}: steps={len(steps)}, has_sublayer={has_sublayer}, has_layer={has_layer}')

    # 1) Layer-wise (aggregate over steps)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=False)
    for ax, (name, title, color) in zip(axes, series):
        if name == 'block' and has_layer:
            x_r, mean_r, min_r, max_r = collect_layerwise_from_layer_metrics(steps, 'para_perp_ratio')
            x_c, mean_c, min_c, max_c = collect_layerwise_from_layer_metrics(steps, 'io_cos_sim')
        elif has_sublayer:
            x_r, mean_r, min_r, max_r = collect_layerwise_from_sublayer(steps, key_map[name], 'para_perp_ratio')
            x_c, mean_c, min_c, max_c = collect_layerwise_from_sublayer(steps, key_map[name], 'io_cos_sim')
        else:
            x_r, mean_r, min_r, max_r = [], [], [], []
            x_c, mean_c, min_c, max_c = [], [], [], []

        if x_r:
            ax.plot(x_r, mean_r, color=color, label='ratio')
            ax.fill_between(x_r, min_r, max_r, color=color, alpha=0.18, label='_nolegend_')
        else:
            ax.text(0.5, 0.63, 'No ratio data', ha='center', va='center', transform=ax.transAxes)

        ax2 = ax.twinx()
        if x_c:
            ax2.plot(x_c, mean_c, color='tab:red', linestyle='--', label='cos')
            ax2.fill_between(x_c, min_c, max_c, color='tab:red', alpha=0.12, label='_nolegend_')
        else:
            ax.text(0.5, 0.42, 'No cos data', ha='center', va='center', transform=ax.transAxes)

        ax.set_title(f'{title} by Layer')
        ax.set_xlabel('Layer')
        ax.set_ylabel('para_perp_ratio', color=color)
        ax.grid(alpha=0.25)
        ax2.set_ylabel('io_cos_sim', color='tab:red')
        ax2.set_ylim(0, 1.05)

        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        if lines1 or lines2:
            ax.legend(lines1 + lines2, labels1 + labels2, ncol=2, loc='center')

    fig.suptitle(f'Prompt {prompt_idx}: layer-wise metrics (aggregated over steps)')
    fig.tight_layout()
    if SAVE_FIG:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = OUT_DIR / f'generation_layerwise_prompt{prompt_idx}.png'
        fig.savefig(out_path, dpi=150)
        print(f'saved: {out_path}')
    plt.show()

    # 2) Step-wise (aggregate over layers)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=False)
    for ax, (name, title, color) in zip(axes, series):
        if name == 'block' and has_layer:
            x_r, mean_r, min_r, max_r = collect_stepwise_from_layer_metrics(steps, 'para_perp_ratio')
            x_c, mean_c, min_c, max_c = collect_stepwise_from_layer_metrics(steps, 'io_cos_sim')
        elif has_sublayer:
            x_r, mean_r, min_r, max_r = collect_stepwise_from_sublayer(steps, key_map[name], 'para_perp_ratio')
            x_c, mean_c, min_c, max_c = collect_stepwise_from_sublayer(steps, key_map[name], 'io_cos_sim')
        else:
            x_r, mean_r, min_r, max_r = [], [], [], []
            x_c, mean_c, min_c, max_c = [], [], [], []

        if x_r:
            ax.plot(x_r, mean_r, color=color, label='ratio')
            ax.fill_between(x_r, min_r, max_r, color=color, alpha=0.18, label='_nolegend_')
        else:
            ax.text(0.5, 0.63, 'No ratio data', ha='center', va='center', transform=ax.transAxes)

        ax2 = ax.twinx()
        if x_c:
            ax2.plot(x_c, mean_c, color='tab:red', linestyle='--', label='cos')
            ax2.fill_between(x_c, min_c, max_c, color='tab:red', alpha=0.12, label='_nolegend_')
        else:
            ax.text(0.5, 0.42, 'No cos data', ha='center', va='center', transform=ax.transAxes)

        ax.set_title(f'{title} by Step')
        ax.set_xlabel('Step')
        ax.set_ylabel('para_perp_ratio', color=color)
        ax.grid(alpha=0.25)
        ax2.set_ylabel('io_cos_sim', color='tab:red')
        ax2.set_ylim(0, 1.05)

        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        if lines1 or lines2:
            ax.legend(lines1 + lines2, labels1 + labels2, ncol=2, loc='center')

    fig.suptitle(f'Prompt {prompt_idx}: step-wise metrics (aggregated over layers)')
    fig.tight_layout()
    if SAVE_FIG:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = OUT_DIR / f'generation_stepwise_prompt{prompt_idx}.png'
        fig.savefig(out_path, dpi=150)
        print(f'saved: {out_path}')
    plt.show()

if runs and not any(any(('sublayer_metrics' in s and s['sublayer_metrics']) for s in (pick_steps(r) or [])) for r in runs):
    print('当前文件没有 sublayer_metrics：attn/mlp 的 ratio/cos 会显示 No data。')
    print('如需 attn/mlp，请重新运行 run_generation_probe.py 并确保 include_sublayer_metrics=True。')



In [ ]:
# # Prompt-aggregated single large figure
# import numpy as np

# PHASE = 'decode'  # 'decode' or 'prefill'
# SAVE_PROMPT_AGG = False


# def pick_steps(run, phase='decode'):
#     timeline = run.get('timeline') or []
#     if timeline:
#         picked = [s for s in timeline if s.get('phase') == phase]
#         return picked if picked else timeline
#     return run.get('steps', [])


# def collect_layerwise_from_layer_metrics(steps, value_key):
#     by_layer = {}
#     for step in steps:
#         for m in (step.get('layer_metrics') or []):
#             l = m.get('layer')
#             v = m.get(value_key)
#             if l is None or v is None:
#                 continue
#             by_layer.setdefault(int(l), []).append(float(v))
#     layers = sorted(by_layer)
#     if not layers:
#         return [], [], [], []
#     means = [sum(by_layer[l]) / len(by_layer[l]) for l in layers]
#     mins = [min(by_layer[l]) for l in layers]
#     maxs = [max(by_layer[l]) for l in layers]
#     return layers, means, mins, maxs


# def collect_layerwise_from_sublayer(steps, metric_key, value_key):
#     by_layer = {}
#     for step in steps:
#         for item in (step.get('sublayer_metrics') or []):
#             l = item.get('layer')
#             v = (item.get(metric_key) or {}).get(value_key)
#             if l is None or v is None:
#                 continue
#             by_layer.setdefault(int(l), []).append(float(v))
#     layers = sorted(by_layer)
#     if not layers:
#         return [], [], [], []
#     means = [sum(by_layer[l]) / len(by_layer[l]) for l in layers]
#     mins = [min(by_layer[l]) for l in layers]
#     maxs = [max(by_layer[l]) for l in layers]
#     return layers, means, mins, maxs


# def merge_across_prompts(series_list):
#     # series_list: list of (layers, means, mins, maxs) from different prompts
#     by_layer = {}
#     for layers, means, mins, maxs in series_list:
#         for l, m, lo, hi in zip(layers, means, mins, maxs):
#             by_layer.setdefault(int(l), {'mean': [], 'min': [], 'max': []})
#             by_layer[int(l)]['mean'].append(float(m))
#             by_layer[int(l)]['min'].append(float(lo))
#             by_layer[int(l)]['max'].append(float(hi))
#     layers = sorted(by_layer)
#     if not layers:
#         return [], [], [], []
#     means = [float(np.mean(by_layer[l]['mean'])) for l in layers]
#     mins = [float(np.min(by_layer[l]['min'])) for l in layers]
#     maxs = [float(np.max(by_layer[l]['max'])) for l in layers]
#     return layers, means, mins, maxs


# parts = [
#     ('block', 'BLOCK', 'tab:blue', None),
#     ('attn', 'ATTN', 'tab:orange', 'attn_update'),
#     ('mlp', 'MLP', 'tab:green', 'mlp_update'),
# ]

# fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=False)

# for ax, (name, title, color, subkey) in zip(axes, parts):
#     ratio_all = []
#     cos_all = []

#     for run in runs:
#         steps = pick_steps(run, phase=PHASE)
#         if not steps:
#             continue
#         if subkey is None:
#             r = collect_layerwise_from_layer_metrics(steps, 'para_perp_ratio')
#             c = collect_layerwise_from_layer_metrics(steps, 'io_cos_sim')
#         else:
#             r = collect_layerwise_from_sublayer(steps, subkey, 'para_perp_ratio')
#             c = collect_layerwise_from_sublayer(steps, subkey, 'io_cos_sim')
#         if r[0]:
#             ratio_all.append(r)
#         if c[0]:
#             cos_all.append(c)

#     x_r, mean_r, min_r, max_r = merge_across_prompts(ratio_all)
#     x_c, mean_c, min_c, max_c = merge_across_prompts(cos_all)

#     if x_r:
#         ax.plot(x_r, mean_r, color=color, label='ratio (prompt-agg mean)')
#         ax.fill_between(x_r, min_r, max_r, color=color, alpha=0.18, label='_nolegend_')
#     else:
#         ax.text(0.5, 0.63, 'No ratio data', ha='center', va='center', transform=ax.transAxes)

#     ax2 = ax.twinx()
#     if x_c:
#         ax2.plot(x_c, mean_c, color='tab:red', linestyle='--', label='cos (prompt-agg mean)')
#         ax2.fill_between(x_c, min_c, max_c, color='tab:red', alpha=0.12, label='_nolegend_')
#     else:
#         ax.text(0.5, 0.42, 'No cos data', ha='center', va='center', transform=ax.transAxes)

#     ax.set_title(f'{title} by Layer')
#     ax.set_xlabel('Layer')
#     ax.set_ylabel('para_perp_ratio', color=color)
#     ax.grid(alpha=0.25)
#     ax2.set_ylabel('io_cos_sim', color='tab:red')
#     ax2.set_ylim(0, 1.05)

#     l1, lb1 = ax.get_legend_handles_labels()
#     l2, lb2 = ax2.get_legend_handles_labels()
#     if l1 or l2:
#         ax.legend(l1 + l2, lb1 + lb2, loc='best', fontsize=8)

# fig.suptitle(f'Prompt-aggregated ({PHASE}) layer-wise metrics')
# fig.tight_layout()

# if SAVE_PROMPT_AGG:
#     OUT_DIR.mkdir(parents=True, exist_ok=True)
#     out = OUT_DIR / f'prompt_agg_{PHASE}_layerwise.png'
#     fig.savefig(out, dpi=160)
#     print('saved:', out)

# plt.show()



In [ ]:
# Per-layer subplots, x-axis=step, aggregated across prompts (mean +/- std)
import math
import numpy as np

PHASE = 'prefill'   # 'decode' or 'prefill'
SAVE_PROMPT_AGG_LAYER_GRID = False


def pick_steps(run, phase='decode'):
    timeline = run.get('timeline') or []
    if timeline:
        picked = [s for s in timeline if s.get('phase') == phase]
        return picked if picked else timeline
    return run.get('steps', [])


def collect_step_layer_values(runs, part='block', metric='para_perp_ratio', phase='decode'):
    # returns: dict[layer][step] -> list(values across prompts)
    out = {}
    for run in runs:
        steps = pick_steps(run, phase=phase)
        for s in steps:
            st = s.get('step')
            if st is None:
                continue
            if part == 'block':
                sub_items = s.get('sublayer_metrics') or []
                if sub_items:
                    for it in sub_items:
                        l = it.get('layer')
                        v = (it.get('block_update') or {}).get(metric)
                        if l is None or v is None:
                            continue
                        out.setdefault(int(l), {}).setdefault(int(st), []).append(float(v))
                else:
                    for m in (s.get('layer_metrics') or []):
                        l = m.get('layer')
                        v = m.get(metric)
                        if l is None or v is None:
                            continue
                        out.setdefault(int(l), {}).setdefault(int(st), []).append(float(v))
            else:
                subkey = 'attn_update' if part == 'attn' else 'mlp_update'
                items = s.get('sublayer_metrics') or []
                for it in items:
                    l = it.get('layer')
                    v = (it.get(subkey) or {}).get(metric)
                    if l is None or v is None:
                        continue
                    out.setdefault(int(l), {}).setdefault(int(st), []).append(float(v))
    return out


def build_mean_min_max_series(step_map):
    steps = sorted(step_map.keys())
    means = [float(np.mean(step_map[s])) for s in steps]
    mins = [float(np.min(step_map[s])) for s in steps]
    maxs = [float(np.max(step_map[s])) for s in steps]
    return steps, means, mins, maxs


def plot_layer_grid_by_step(runs, part, phase='decode', save_name=None):
    ratio = collect_step_layer_values(runs, part=part, metric='para_perp_ratio', phase=phase)
    cos = collect_step_layer_values(runs, part=part, metric='io_cos_sim', phase=phase)

    layers = sorted(set(ratio.keys()) | set(cos.keys()))
    if not layers:
        print(f'{part}: no data')
        return

    n = len(layers)
    ncols = 6
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.6 * nrows), sharex=False)
    axes = np.array(axes).reshape(-1)

    for i, l in enumerate(layers):
        ax = axes[i]
        ax2 = ax.twinx()

        if l in ratio and ratio[l]:
            x, m, lo, hi = build_mean_min_max_series(ratio[l])
            m = np.array(m)
            lo = np.array(lo)
            hi = np.array(hi)
            ax.plot(x, m, color='tab:blue', lw=1.2, label='ratio mean')
            ax.fill_between(x, lo, hi, color='tab:blue', alpha=0.18, label='ratio min-max')
        else:
            ax.text(0.5, 0.62, 'No ratio', ha='center', va='center', transform=ax.transAxes, fontsize=8)

        if l in cos and cos[l]:
            x2, m2, lo2, hi2 = build_mean_min_max_series(cos[l])
            m2 = np.array(m2)
            lo2 = np.array(lo2)
            hi2 = np.array(hi2)
            ax2.plot(x2, m2, color='tab:red', lw=1.1, linestyle='--', label='cos mean')
            ax2.fill_between(x2, lo2, hi2, color='tab:red', alpha=0.12, label='cos min-max')
        else:
            ax.text(0.5, 0.43, 'No cos', ha='center', va='center', transform=ax.transAxes, fontsize=8)

        ax.set_title(f'Layer {l}', fontsize=9)
        ax.set_xlabel('step', fontsize=8)
        ax.set_ylabel('ratio', fontsize=8, color='tab:blue')
        ax2.set_ylabel('cos', fontsize=8, color='tab:red')
        ax2.set_ylim(0, 1.05)
        ax.grid(alpha=0.2)

    for j in range(len(layers), len(axes)):
        axes[j].axis('off')

    fig.suptitle(f'{part.upper()} per-layer (x=step), prompt-aggregated mean + min-max, phase={phase}')
    fig.tight_layout()
    if SAVE_PROMPT_AGG_LAYER_GRID and save_name is not None:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        out = OUT_DIR / save_name
        fig.savefig(out, dpi=160)
        print('saved:', out)
    plt.show()


plot_layer_grid_by_step(runs, part='block', phase=PHASE, save_name=f'prompt_agg_{PHASE}_block_layergrid_step.png')
plot_layer_grid_by_step(runs, part='attn',  phase=PHASE, save_name=f'prompt_agg_{PHASE}_attn_layergrid_step.png')
plot_layer_grid_by_step(runs, part='mlp',   phase=PHASE, save_name=f'prompt_agg_{PHASE}_mlp_layergrid_step.png')





In [ ]:
# Prompt-aggregated visualization (self-contained)
from pathlib import Path
import json
import math
import numpy as np
import matplotlib.pyplot as plt

# ---- config ----
RESULT_PATH = Path('../results/probe.json')
PHASE = 'prefill'   # 'decode' or 'prefill'
SAVE_PROMPT_AGG_LAYER_GRID = False
OUT_DIR = Path('../results')

# ---- load ----
data = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
runs = data['runs'] if 'runs' in data else [data]
print(f'loaded runs: {len(runs)} from {RESULT_PATH}')


def pick_steps(run, phase='decode'):
    timeline = run.get('timeline') or []
    if timeline:
        picked = [s for s in timeline if s.get('phase') == phase]
        return picked if picked else timeline
    return run.get('steps', [])


def collect_step_layer_values(runs, part='block', metric='para_perp_ratio', phase='decode'):
    # returns: dict[layer][step] -> list(values across prompts)
    out = {}
    for run in runs:
        steps = pick_steps(run, phase=phase)
        for s in steps:
            st = s.get('step')
            if st is None:
                continue

            if part == 'block':
                # Prefer sublayer block_update; fallback to layer_metrics.
                sub_items = s.get('sublayer_metrics') or []
                if sub_items:
                    for it in sub_items:
                        l = it.get('layer')
                        v = (it.get('block_update') or {}).get(metric)
                        if l is None or v is None:
                            continue
                        out.setdefault(int(l), {}).setdefault(int(st), []).append(float(v))
                else:
                    for m in (s.get('layer_metrics') or []):
                        l = m.get('layer')
                        v = m.get(metric)
                        if l is None or v is None:
                            continue
                        out.setdefault(int(l), {}).setdefault(int(st), []).append(float(v))
            else:
                subkey = 'attn_update' if part == 'attn' else 'mlp_update'
                for it in (s.get('sublayer_metrics') or []):
                    l = it.get('layer')
                    v = (it.get(subkey) or {}).get(metric)
                    if l is None or v is None:
                        continue
                    out.setdefault(int(l), {}).setdefault(int(st), []).append(float(v))
    return out


def build_mean_min_max_series(step_map):
    steps = sorted(step_map.keys())
    means = [float(np.mean(step_map[s])) for s in steps]
    mins = [float(np.min(step_map[s])) for s in steps]
    maxs = [float(np.max(step_map[s])) for s in steps]
    return steps, means, mins, maxs


def plot_layer_grid_by_step(runs, part, phase='decode', save_name=None):
    ratio = collect_step_layer_values(runs, part=part, metric='para_perp_ratio', phase=phase)
    cos = collect_step_layer_values(runs, part=part, metric='io_cos_sim', phase=phase)

    layers = sorted(set(ratio.keys()) | set(cos.keys()))
    if not layers:
        print(f'{part}: no data')
        return

    n = len(layers)
    ncols = 6
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 2.6 * nrows), sharex=False)
    axes = np.array(axes).reshape(-1)

    for i, l in enumerate(layers):
        ax = axes[i]
        ax2 = ax.twinx()

        if l in ratio and ratio[l]:
            x, m, lo, hi = build_mean_min_max_series(ratio[l])
            m = np.array(m)
            lo = np.array(lo)
            hi = np.array(hi)
            ax.plot(x, m, color='tab:blue', lw=1.2, label='ratio mean')
            ax.fill_between(x, lo, hi, color='tab:blue', alpha=0.18, label='ratio min-max')
        else:
            ax.text(0.5, 0.62, 'No ratio', ha='center', va='center', transform=ax.transAxes, fontsize=8)

        if l in cos and cos[l]:
            x2, m2, lo2, hi2 = build_mean_min_max_series(cos[l])
            m2 = np.array(m2)
            lo2 = np.array(lo2)
            hi2 = np.array(hi2)
            ax2.plot(x2, m2, color='tab:red', lw=1.1, linestyle='--', label='cos mean')
            ax2.fill_between(x2, lo2, hi2, color='tab:red', alpha=0.12, label='cos min-max')
        else:
            ax.text(0.5, 0.43, 'No cos', ha='center', va='center', transform=ax.transAxes, fontsize=8)

        ax.set_title(f'Layer {l}', fontsize=9)
        ax.set_xlabel('step', fontsize=8)
        ax.set_ylabel('ratio', fontsize=8, color='tab:blue')
        ax2.set_ylabel('cos', fontsize=8, color='tab:red')
        ax2.set_ylim(0, 1.05)
        ax.grid(alpha=0.2)

    for j in range(len(layers), len(axes)):
        axes[j].axis('off')

    fig.suptitle(f'{part.upper()} per-layer (x=step), prompt-aggregated mean + min-max, phase={phase}')
    fig.tight_layout()

    if SAVE_PROMPT_AGG_LAYER_GRID and save_name is not None:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        out = OUT_DIR / save_name
        fig.savefig(out, dpi=160)
        print('saved:', out)

    plt.show()


plot_layer_grid_by_step(runs, part='block', phase=PHASE, save_name=f'prompt_agg_{PHASE}_block_layergrid_step.png')
plot_layer_grid_by_step(runs, part='attn',  phase=PHASE, save_name=f'prompt_agg_{PHASE}_attn_layergrid_step.png')
plot_layer_grid_by_step(runs, part='mlp',   phase=PHASE, save_name=f'prompt_agg_{PHASE}_mlp_layergrid_step.png')



In [ ]:
# NEW BLOCK: per-prompt plots, x-axis=layer, range over steps (min-max)
from pathlib import Path
import json
import matplotlib.pyplot as plt

# ---- config ----
# RESULT_PATH = Path('../results/generation_probe.json')
PHASE = 'decode'   # 'decode' or 'prefill'
SAVE_FIG = False
OUT_DIR = Path('../results')
LEFT_METRIC = 'dz_perp_over_z_plus_dz_para'  # or 'para_perp_ratio'
LEFT_LABEL = 'dz_perp/(z+dz_para)' if LEFT_METRIC == 'dz_perp_over_z_plus_dz_para' else 'para_perp_ratio'
TRIM_EDGE_LAYERS = False
# -----------------

data = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
runs = data['runs'] if 'runs' in data else [data]
print(f'loaded runs: {len(runs)} from {RESULT_PATH}')


def pick_steps(run, phase='decode'):
    timeline = run.get('timeline') or []
    if timeline:
        picked = [s for s in timeline if s.get('phase') == phase]
        return picked if picked else timeline
    if phase == 'prefill':
        prefill = run.get('prefill_steps') or []
        if prefill:
            return prefill
    return run.get('steps', [])


def collect_layerwise_from_layer_metrics(steps, value_key):
    by_layer = {}
    for s in steps:
        for m in (s.get('layer_metrics') or []):
            l = m.get('layer')
            v = m.get(value_key)
            if l is None or v is None:
                continue
            by_layer.setdefault(int(l), []).append(float(v))
    layers = sorted(by_layer)
    if TRIM_EDGE_LAYERS and len(layers) > 2:
        layers = layers[1:-1]
    means = [sum(by_layer[l]) / len(by_layer[l]) for l in layers]
    mins = [min(by_layer[l]) for l in layers]
    maxs = [max(by_layer[l]) for l in layers]
    return layers, means, mins, maxs


def collect_layerwise_from_sublayer(steps, metric_key, value_key):
    by_layer = {}
    for s in steps:
        for item in (s.get('sublayer_metrics') or []):
            l = item.get('layer')
            v = (item.get(metric_key) or {}).get(value_key)
            if l is None or v is None:
                continue
            by_layer.setdefault(int(l), []).append(float(v))
    layers = sorted(by_layer)
    if TRIM_EDGE_LAYERS and len(layers) > 2:
        layers = layers[1:-1]
    means = [sum(by_layer[l]) / len(by_layer[l]) for l in layers]
    mins = [min(by_layer[l]) for l in layers]
    maxs = [max(by_layer[l]) for l in layers]
    return layers, means, mins, maxs


series = [('block', 'BLOCK', 'block_update'), ('attn', 'ATTN', 'attn_update'), ('mlp', 'MLP', 'mlp_update')]

for prompt_idx, run in enumerate(runs):
    steps = pick_steps(run, phase=PHASE)
    if not steps:
        print(f'prompt {prompt_idx}: no steps ({PHASE}), skipped')
        continue

    has_sublayer = any(('sublayer_metrics' in s and s['sublayer_metrics']) for s in steps)
    has_layer = any(('layer_metrics' in s and s['layer_metrics']) for s in steps)
    print(f'prompt {prompt_idx}: steps={len(steps)}, phase={PHASE}, has_sublayer={has_sublayer}, has_layer={has_layer}')

    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=False)

    for ax, (name, title, metric_key) in zip(axes, series):
        used_layer_left_fallback = False
        used_layer_cos_fallback = False

        if has_sublayer:
            if name == 'block':
                # block: prefer block_update in sublayer metrics
                x_l, mean_l, min_l, max_l = collect_layerwise_from_sublayer(steps, 'block_update', LEFT_METRIC)
                x_c, mean_c, min_c, max_c = collect_layerwise_from_sublayer(steps, 'block_update', 'io_cos_sim')
                if has_layer and not x_l:
                    x_l, mean_l, min_l, max_l = collect_layerwise_from_layer_metrics(steps, LEFT_METRIC)
                    used_layer_left_fallback = True
                if has_layer and not x_c:
                    x_c, mean_c, min_c, max_c = collect_layerwise_from_layer_metrics(steps, 'io_cos_sim')
                    used_layer_cos_fallback = True
            else:
                x_l, mean_l, min_l, max_l = collect_layerwise_from_sublayer(steps, metric_key, LEFT_METRIC)
                x_c, mean_c, min_c, max_c = collect_layerwise_from_sublayer(steps, metric_key, 'io_cos_sim')
        else:
            x_l, mean_l, min_l, max_l = collect_layerwise_from_layer_metrics(steps, LEFT_METRIC)
            x_c, mean_c, min_c, max_c = collect_layerwise_from_layer_metrics(steps, 'io_cos_sim')

        if x_l:
            ax.plot(x_l, mean_l, color='tab:blue', linestyle='-.', label=f'{LEFT_LABEL}')
            ax.fill_between(x_l, min_l, max_l, color='tab:blue', alpha=0.15, label='_nolegend_')
        else:
            ax.text(0.5, 0.62, f'No {LEFT_LABEL} data', ha='center', va='center', transform=ax.transAxes)

        ax2 = ax.twinx()
        if x_c:
            ax2.plot(x_c, mean_c, color='tab:red', linestyle='--', label='io_cos_sim')
            ax2.fill_between(x_c, min_c, max_c, color='tab:red', alpha=0.12, label='_nolegend_')
        else:
            ax.text(0.5, 0.42, 'No cos data', ha='center', va='center', transform=ax.transAxes)

        ax.set_title(f'{title} (range over steps)')
        ax.set_xlabel('Layer')
        ax.set_ylabel(LEFT_LABEL, color='tab:blue')
        ax.grid(alpha=0.25)

        ax2.set_ylabel('io_cos_sim', color='tab:red')
        ax2.set_ylim(0, 1.05)

        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        if lines1 or lines2:
            ax.legend(lines1 + lines2, labels1 + labels2, ncol=2, loc='center')

        if used_layer_left_fallback:
            ax.text(0.01, 0.02, f'{LEFT_LABEL} from layer_metrics', transform=ax.transAxes, fontsize=8, color='tab:blue')
        if used_layer_cos_fallback:
            ax.text(0.01, 0.08, 'cos from layer_metrics', transform=ax.transAxes, fontsize=8, color='tab:red')

    fig.suptitle(f'Prompt {prompt_idx}: layer-wise metrics (range over steps, phase={PHASE})')
    fig.tight_layout()

    if SAVE_FIG:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = OUT_DIR / f'prompt{prompt_idx}_{PHASE}_layerwise_range.png'
        fig.savefig(out_path, dpi=150)
        print(f'saved: {out_path}')

    plt.show()



In [ ]:
# NEW BLOCK: per-prompt layer-wise plot with z norm + io_cos_sim (range over steps)
from pathlib import Path
import json
import matplotlib.pyplot as plt

# ---- config ----
# RESULT_PATH = Path('../results/generation_probe.json')
PHASE = 'decode'   # 'decode' or 'prefill'
SAVE_FIG = False
OUT_DIR = Path('../results')
TRIM_EDGE_LAYERS = False
# -----------------

data = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
runs = data['runs'] if 'runs' in data else [data]
print(f'loaded runs: {len(runs)} from {RESULT_PATH}')


def pick_steps(run, phase='decode'):
    timeline = run.get('timeline') or []
    if timeline:
        picked = [s for s in timeline if s.get('phase') == phase]
        return picked if picked else timeline
    if phase == 'prefill':
        prefill = run.get('prefill_steps') or []
        if prefill:
            return prefill
    return run.get('steps', [])


def collect_layerwise_from_layer_metrics(steps, value_key):
    by_layer = {}
    for s in steps:
        for m in (s.get('layer_metrics') or []):
            l = m.get('layer')
            v = m.get(value_key)
            if l is None or v is None:
                continue
            by_layer.setdefault(int(l), []).append(float(v))
    layers = sorted(by_layer)
    if TRIM_EDGE_LAYERS and len(layers) > 2:
        layers = layers[1:-1]
    means = [sum(by_layer[l]) / len(by_layer[l]) for l in layers]
    mins = [min(by_layer[l]) for l in layers]
    maxs = [max(by_layer[l]) for l in layers]
    return layers, means, mins, maxs


def collect_layerwise_from_sublayer(steps, metric_key, value_key):
    by_layer = {}
    for s in steps:
        for item in (s.get('sublayer_metrics') or []):
            l = item.get('layer')
            v = (item.get(metric_key) or {}).get(value_key)
            if l is None or v is None:
                continue
            by_layer.setdefault(int(l), []).append(float(v))
    layers = sorted(by_layer)
    if TRIM_EDGE_LAYERS and len(layers) > 2:
        layers = layers[1:-1]
    means = [sum(by_layer[l]) / len(by_layer[l]) for l in layers]
    mins = [min(by_layer[l]) for l in layers]
    maxs = [max(by_layer[l]) for l in layers]
    return layers, means, mins, maxs


series = [('block', 'BLOCK', 'block_update'), ('attn', 'ATTN', 'attn_update'), ('mlp', 'MLP', 'mlp_update')]

for prompt_idx, run in enumerate(runs):
    steps = pick_steps(run, phase=PHASE)
    if not steps:
        print(f'prompt {prompt_idx}: no steps ({PHASE}), skipped')
        continue

    has_sublayer = any(('sublayer_metrics' in s and s['sublayer_metrics']) for s in steps)
    has_layer = any(('layer_metrics' in s and s['layer_metrics']) for s in steps)
    print(f'prompt {prompt_idx}: steps={len(steps)}, phase={PHASE}, has_sublayer={has_sublayer}, has_layer={has_layer}')

    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=False)

    for ax, (name, title, metric_key) in zip(axes, series):
        used_layer_z_fallback = False
        used_layer_cos_fallback = False

        if has_sublayer:
            if name == 'block':
                x_z, mean_z, min_z, max_z = collect_layerwise_from_sublayer(steps, 'block_update', 'z_post_norm')
                x_c, mean_c, min_c, max_c = collect_layerwise_from_sublayer(steps, 'block_update', 'io_cos_sim')
                if has_layer and not x_z:
                    x_z, mean_z, min_z, max_z = collect_layerwise_from_layer_metrics(steps, 'z_post_norm')
                    used_layer_z_fallback = True
                if has_layer and not x_c:
                    x_c, mean_c, min_c, max_c = collect_layerwise_from_layer_metrics(steps, 'io_cos_sim')
                    used_layer_cos_fallback = True
            else:
                x_z, mean_z, min_z, max_z = collect_layerwise_from_sublayer(steps, metric_key, 'z_post_norm')
                x_c, mean_c, min_c, max_c = collect_layerwise_from_sublayer(steps, metric_key, 'io_cos_sim')
        else:
            x_z, mean_z, min_z, max_z = collect_layerwise_from_layer_metrics(steps, 'z_post_norm')
            x_c, mean_c, min_c, max_c = collect_layerwise_from_layer_metrics(steps, 'io_cos_sim')

        if x_z:
            ax.plot(x_z, mean_z, color='tab:blue', linestyle='-.', label='z_post_norm')
            ax.fill_between(x_z, min_z, max_z, color='tab:blue', alpha=0.15, label='_nolegend_')
        else:
            ax.text(0.5, 0.62, 'No z norm data', ha='center', va='center', transform=ax.transAxes)

        ax2 = ax.twinx()
        if x_c:
            ax2.plot(x_c, mean_c, color='tab:red', linestyle='--', label='io_cos_sim')
            ax2.fill_between(x_c, min_c, max_c, color='tab:red', alpha=0.12, label='_nolegend_')
        else:
            ax.text(0.5, 0.42, 'No cos data', ha='center', va='center', transform=ax.transAxes)

        ax.set_title(f'{title} (range over steps)')
        ax.set_xlabel('Layer')
        ax.set_ylabel('z_post_norm', color='tab:blue')
        ax.grid(alpha=0.25)

        ax2.set_ylabel('io_cos_sim', color='tab:red')
        ax2.set_ylim(0, 1.05)

        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        if lines1 or lines2:
            ax.legend(lines1 + lines2, labels1 + labels2, ncol=2, loc='center')

        if used_layer_z_fallback:
            ax.text(0.01, 0.02, 'z norm from layer_metrics', transform=ax.transAxes, fontsize=8, color='tab:blue')
        if used_layer_cos_fallback:
            ax.text(0.01, 0.08, 'cos from layer_metrics', transform=ax.transAxes, fontsize=8, color='tab:red')

    fig.suptitle(f'Prompt {prompt_idx}: z norm + io_cos_sim (range over steps, phase={PHASE})')
    fig.tight_layout()

    if SAVE_FIG:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = OUT_DIR / f'prompt{prompt_idx}_{PHASE}_layerwise_znorm_range.png'
        fig.savefig(out_path, dpi=150)
        print(f'saved: {out_path}')

    plt.show()



In [ ]:
# NEW BLOCK: z norm + dz norm (range over steps), per prompt, layer-wise
from pathlib import Path
import json
import matplotlib.pyplot as plt

# ---- config ----
# RESULT_PATH = Path('../results/generation_probe.json')
PHASE = 'decode'   # 'decode' or 'prefill'
SAVE_FIG = False
OUT_DIR = Path('../results')
TRIM_EDGE_LAYERS = False
# -----------------

data = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
runs = data['runs'] if 'runs' in data else [data]
print(f'loaded runs: {len(runs)} from {RESULT_PATH}')


def pick_steps(run, phase='decode'):
    timeline = run.get('timeline') or []
    if timeline:
        picked = [s for s in timeline if s.get('phase') == phase]
        return picked if picked else timeline
    if phase == 'prefill':
        prefill = run.get('prefill_steps') or []
        if prefill:
            return prefill
    return run.get('steps', [])


def collect_layerwise_from_layer_metrics(steps, value_key):
    by_layer = {}
    for s in steps:
        for m in (s.get('layer_metrics') or []):
            l = m.get('layer')
            v = m.get(value_key)
            if l is None or v is None:
                continue
            by_layer.setdefault(int(l), []).append(float(v))
    layers = sorted(by_layer)
    if TRIM_EDGE_LAYERS and len(layers) > 2:
        layers = layers[1:-1]
    means = [sum(by_layer[l]) / len(by_layer[l]) for l in layers]
    mins = [min(by_layer[l]) for l in layers]
    maxs = [max(by_layer[l]) for l in layers]
    return layers, means, mins, maxs


def collect_layerwise_from_sublayer(steps, metric_key, value_key):
    by_layer = {}
    for s in steps:
        for item in (s.get('sublayer_metrics') or []):
            l = item.get('layer')
            v = (item.get(metric_key) or {}).get(value_key)
            if l is None or v is None:
                continue
            by_layer.setdefault(int(l), []).append(float(v))
    layers = sorted(by_layer)
    if TRIM_EDGE_LAYERS and len(layers) > 2:
        layers = layers[1:-1]
    means = [sum(by_layer[l]) / len(by_layer[l]) for l in layers]
    mins = [min(by_layer[l]) for l in layers]
    maxs = [max(by_layer[l]) for l in layers]
    return layers, means, mins, maxs


series = [('block', 'BLOCK', 'block_update'), ('attn', 'ATTN', 'attn_update'), ('mlp', 'MLP', 'mlp_update')]

for prompt_idx, run in enumerate(runs):
    steps = pick_steps(run, phase=PHASE)
    if not steps:
        print(f'prompt {prompt_idx}: no steps ({PHASE}), skipped')
        continue

    has_sublayer = any(('sublayer_metrics' in s and s['sublayer_metrics']) for s in steps)
    has_layer = any(('layer_metrics' in s and s['layer_metrics']) for s in steps)
    print(f'prompt {prompt_idx}: steps={len(steps)}, phase={PHASE}, has_sublayer={has_sublayer}, has_layer={has_layer}')

    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=False)

    for ax, (name, title, metric_key) in zip(axes, series):
        used_layer_z_fallback = False
        used_layer_dz_fallback = False

        if has_sublayer:
            if name == 'block':
                x_z, mean_z, min_z, max_z = collect_layerwise_from_sublayer(steps, 'block_update', 'z_post_norm')
                x_dz, mean_dz, min_dz, max_dz = collect_layerwise_from_sublayer(steps, 'block_update', 'dz_norm')
                if has_layer and not x_z:
                    x_z, mean_z, min_z, max_z = collect_layerwise_from_layer_metrics(steps, 'z_post_norm')
                    used_layer_z_fallback = True
                if has_layer and not x_dz:
                    x_dz, mean_dz, min_dz, max_dz = collect_layerwise_from_layer_metrics(steps, 'dz_norm')
                    used_layer_dz_fallback = True
            else:
                x_z, mean_z, min_z, max_z = collect_layerwise_from_sublayer(steps, metric_key, 'z_post_norm')
                x_dz, mean_dz, min_dz, max_dz = collect_layerwise_from_sublayer(steps, metric_key, 'dz_norm')
        else:
            x_z, mean_z, min_z, max_z = collect_layerwise_from_layer_metrics(steps, 'z_post_norm')
            x_dz, mean_dz, min_dz, max_dz = collect_layerwise_from_layer_metrics(steps, 'dz_norm')

        if x_z:
            ax.plot(x_z, mean_z, color='tab:blue', linestyle='-.', label='z_post_norm')
            ax.fill_between(x_z, min_z, max_z, color='tab:blue', alpha=0.15, label='_nolegend_')
        else:
            ax.text(0.5, 0.62, 'No z norm data', ha='center', va='center', transform=ax.transAxes)

        ax2 = ax.twinx()
        if x_dz:
            ax2.plot(x_dz, mean_dz, color='tab:orange', linestyle='--', label='dz_norm')
            ax2.fill_between(x_dz, min_dz, max_dz, color='tab:orange', alpha=0.12, label='_nolegend_')
        else:
            ax.text(0.5, 0.42, 'No dz norm data', ha='center', va='center', transform=ax.transAxes)

        ax.set_title(f'{title} (range over steps)')
        ax.set_xlabel('Layer')
        ax.set_ylabel('z_post_norm', color='tab:blue')
        ax.grid(alpha=0.25)

        ax2.set_ylabel('dz_norm', color='tab:orange')

        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        if lines1 or lines2:
            ax.legend(lines1 + lines2, labels1 + labels2, ncol=2, loc='center')

        if used_layer_z_fallback:
            ax.text(0.01, 0.02, 'z norm from layer_metrics', transform=ax.transAxes, fontsize=8, color='tab:blue')
        if used_layer_dz_fallback:
            ax.text(0.01, 0.08, 'dz norm from layer_metrics', transform=ax.transAxes, fontsize=8, color='tab:orange')

    fig.suptitle(f'Prompt {prompt_idx}: z norm + dz norm (range over steps, phase={PHASE})')
    fig.tight_layout()

    if SAVE_FIG:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = OUT_DIR / f'prompt{prompt_idx}_{PHASE}_layerwise_znorm_dznorm_range.png'
        fig.savefig(out_path, dpi=150)
        print(f'saved: {out_path}')

    plt.show()



In [ ]:
# Single-group visualization from one JSON (no base/instruct split)
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

RESULT_PATH = Path('results/generation_probe.json')
PHASE = 'decode'   # 'prefill' | 'decode'
SAVE_FIG = False
OUT_DIR = Path('../results')

SUMMARY_RATIO_KEY_CANDIDATES = [
    'dz_para_dz_perp_ratio_mean',
    'dz_para_dz_perp_ratio_median',
    'dz_perp_over_z_plus_dz_para_mean',
    'dz_perp_over_z_plus_dz_para_median',
]
SUMMARY_COS_KEY_CANDIDATES = ['io_cos_sim_mean', 'io_cos_sim_median']

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fcfcfc',
    'axes.edgecolor': '#dddddd',
    'axes.titleweight': 'semibold',
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'grid.color': '#d9d9d9',
    'grid.alpha': 0.35,
    'grid.linestyle': '--',
})


def load_runs(path: Path):
    data = json.loads(path.read_text(encoding='utf-8'))
    return data['runs'] if 'runs' in data else [data]


def pick_steps(run, phase='decode'):
    timeline = run.get('timeline') or []
    if timeline:
        picked = [s for s in timeline if s.get('phase') == phase]
        return picked if picked else timeline
    if phase == 'prefill' and run.get('prefill_steps'):
        return run.get('prefill_steps') or []
    return run.get('steps', [])


def has_layerwise_metrics(runs):
    for run in runs:
        for s in pick_steps(run, phase=PHASE):
            if s.get('layer_metrics'):
                return True
            if s.get('sublayer_metrics'):
                return True
    return False


def collect_layerwise_from_layer_metrics(steps, value_key):
    by_layer = {}
    for step in steps:
        for m in (step.get('layer_metrics') or []):
            layer = m.get('layer')
            v = m.get(value_key)
            if layer is None or v is None:
                continue
            by_layer.setdefault(int(layer), []).append(float(v))
    layers = sorted(by_layer.keys())
    if len(layers) > 2:
        layers = layers[1:-1]
    means = [float(np.mean(by_layer[l])) for l in layers]
    mins = [float(np.min(by_layer[l])) for l in layers]
    maxs = [float(np.max(by_layer[l])) for l in layers]
    return layers, means, mins, maxs


def collect_layerwise_from_sublayer(steps, metric_key, value_key):
    by_layer = {}
    for step in steps:
        for item in (step.get('sublayer_metrics') or []):
            layer = item.get('layer')
            v = (item.get(metric_key) or {}).get(value_key)
            if layer is None or v is None:
                continue
            by_layer.setdefault(int(layer), []).append(float(v))
    layers = sorted(by_layer.keys())
    if len(layers) > 2:
        layers = layers[1:-1]
    means = [float(np.mean(by_layer[l])) for l in layers]
    mins = [float(np.min(by_layer[l])) for l in layers]
    maxs = [float(np.max(by_layer[l])) for l in layers]
    return layers, means, mins, maxs


def aggregate_over_prompts_layerwise(runs, part, metric):
    key_map = {'block': 'block_update', 'attn': 'attn_update', 'mlp': 'mlp_update'}
    all_layers = set()
    per_prompt = []

    for run in runs:
        steps = pick_steps(run, phase=PHASE)
        has_sublayer = any(('sublayer_metrics' in s and s['sublayer_metrics']) for s in steps)
        has_layer = any(('layer_metrics' in s and s['layer_metrics']) for s in steps)

        if part == 'block' and has_layer:
            x, m, _, _ = collect_layerwise_from_layer_metrics(steps, metric)
        elif has_sublayer:
            x, m, _, _ = collect_layerwise_from_sublayer(steps, key_map[part], metric)
            if part == 'block' and (not x) and has_layer:
                x, m, _, _ = collect_layerwise_from_layer_metrics(steps, metric)
        else:
            x, m = [], []

        if x:
            all_layers.update(x)
            per_prompt.append((x, m))

    if not all_layers:
        return [], [], [], []

    layers = sorted(all_layers)
    values_by_layer = {l: [] for l in layers}
    for x, m in per_prompt:
        m_by_layer = {lx: mv for lx, mv in zip(x, m)}
        for l in layers:
            if l in m_by_layer:
                values_by_layer[l].append(m_by_layer[l])

    means = [float(np.mean(values_by_layer[l])) if values_by_layer[l] else np.nan for l in layers]
    mins = [float(np.min(values_by_layer[l])) if values_by_layer[l] else np.nan for l in layers]
    maxs = [float(np.max(values_by_layer[l])) if values_by_layer[l] else np.nan for l in layers]
    return layers, means, mins, maxs


def _pick_summary_value(summary, key_candidates):
    for k in key_candidates:
        v = summary.get(k)
        if v is not None:
            return float(v)
    return None


def _legend_key_name(cands):
    return cands[0] if cands else 'metric'


def aggregate_over_prompts_summary(runs, key_candidates):
    all_steps = set()
    per_prompt = []

    for run in runs:
        xs, ys = [], []
        for s in pick_steps(run, phase=PHASE):
            st = s.get('timestep', s.get('step'))
            summary = s.get('summary') or {}
            val = _pick_summary_value(summary, key_candidates)
            if st is None or val is None:
                continue
            xs.append(int(st))
            ys.append(float(val))
        if xs:
            all_steps.update(xs)
            per_prompt.append((xs, ys))

    if not all_steps:
        return [], [], [], []

    steps = sorted(all_steps)
    values_by_step = {s: [] for s in steps}
    for xs, ys in per_prompt:
        d = {x: y for x, y in zip(xs, ys)}
        for s in steps:
            if s in d:
                values_by_step[s].append(d[s])

    means = [float(np.mean(values_by_step[s])) if values_by_step[s] else np.nan for s in steps]
    mins = [float(np.min(values_by_step[s])) if values_by_step[s] else np.nan for s in steps]
    maxs = [float(np.max(values_by_step[s])) if values_by_step[s] else np.nan for s in steps]
    return steps, means, mins, maxs


def plot_layerwise_mode(runs, metric_left='dz_perp_over_z_plus_dz_para', metric_right='io_cos_sim'):
    series = [('block', 'BLOCK'), ('attn', 'ATTN'), ('mlp', 'MLP')]
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.6), sharex=False)

    handles, labels = [], []
    for ax, (part, title) in zip(axes, series):
        lx, lm, llo, lhi = aggregate_over_prompts_layerwise(runs, part, metric_left)
        rx, rm, rlo, rhi = aggregate_over_prompts_layerwise(runs, part, metric_right)

        if lx:
            h = ax.plot(lx, lm, color='#1f77b4', lw=2.0, label='ratio')[0]
            ax.fill_between(lx, llo, lhi, color='#1f77b4', alpha=0.15)
            handles.append(h); labels.append('ratio')

        ax2 = ax.twinx()
        if rx:
            h = ax2.plot(rx, rm, color='#d62728', ls='--', lw=1.8, label='cos')[0]
            ax2.fill_between(rx, rlo, rhi, color='#d62728', alpha=0.11)
            handles.append(h); labels.append('cos')

        ax.set_title(f'{title}')
        ax.set_xlabel('Layer')
        ax.set_ylabel(metric_left)
        ax.grid(True)
        ax2.set_ylabel(metric_right)
        ax2.set_ylim(0, 1.05)

    uniq = {}
    for h, l in zip(handles, labels):
        if l not in uniq:
            uniq[l] = h

    fig.suptitle(f'All runs | {PHASE} phase | layer-wise mode', y=0.98, fontsize=14, fontweight='bold')
    fig.legend(list(uniq.values()), list(uniq.keys()), loc='lower center', bbox_to_anchor=(0.5, 0.03), ncol=2, frameon=False)
    fig.text(0.5, 0.005, 'Solid: ratio | Dashed: io_cos_sim | Shaded: prompt min-max range', ha='center', va='bottom', fontsize=10, color='#555')
    fig.tight_layout(rect=[0.02, 0.09, 0.98, 0.93])

    if SAVE_FIG:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = OUT_DIR / f'all_runs_{PHASE}_layerwise.png'
        fig.savefig(out_path, dpi=180)
        print('saved:', out_path)
    plt.show()


def plot_summary_mode(runs):
    ratio_keys = SUMMARY_RATIO_KEY_CANDIDATES
    cos_keys = SUMMARY_COS_KEY_CANDIDATES

    x, m, lo, hi = aggregate_over_prompts_summary(runs, ratio_keys)
    xc, mc, lco, hco = aggregate_over_prompts_summary(runs, cos_keys)

    fig, ax = plt.subplots(1, 1, figsize=(10.5, 5.4))
    handles, labels = [], []

    if x:
        h = ax.plot(x, m, color='#1f77b4', lw=2.1, label='ratio')[0]
        ax.fill_between(x, lo, hi, color='#1f77b4', alpha=0.15)
        handles.append(h); labels.append('ratio')

    ax2 = ax.twinx()
    if xc:
        h = ax2.plot(xc, mc, color='#d62728', ls='--', lw=1.9, label='cos')[0]
        ax2.fill_between(xc, lco, hco, color='#d62728', alpha=0.11)
        handles.append(h); labels.append('cos')

    ax.set_xlabel('Step')
    ax.set_ylabel(_legend_key_name(SUMMARY_RATIO_KEY_CANDIDATES))
    ax2.set_ylabel(_legend_key_name(SUMMARY_COS_KEY_CANDIDATES))
    ax2.set_ylim(0, 1.05)
    ax.grid(True)

    uniq = {}
    for h, l in zip(handles, labels):
        if l not in uniq:
            uniq[l] = h

    fig.suptitle(f'All runs | {PHASE} phase | summary-only mode', y=0.98, fontsize=14, fontweight='bold')
    fig.legend(list(uniq.values()), list(uniq.keys()), loc='lower center', bbox_to_anchor=(0.5, 0.03), ncol=2, frameon=False)
    fig.text(0.5, 0.005, 'Per-step summary curves aggregated across prompts (mean + min-max range)', ha='center', va='bottom', fontsize=10, color='#555')
    fig.tight_layout(rect=[0.02, 0.09, 0.98, 0.93])

    if SAVE_FIG:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = OUT_DIR / f'all_runs_{PHASE}_summary_only.png'
        fig.savefig(out_path, dpi=180)
        print('saved:', out_path)
    plt.show()


runs = load_runs(RESULT_PATH)
print(f'all runs: {len(runs)} from {RESULT_PATH}')

if has_layerwise_metrics(runs):
    print('mode: layer-wise')
    plot_layerwise_mode(runs)
else:
    print('mode: summary-only (no layer_metrics/sublayer_metrics found)')
    plot_summary_mode(runs)


In [ ]:
# NEW BLOCK: z_post_norm / dz_norm with variance band (per prompt, layer-wise)
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# ---- config ----
# RESULT_PATH = Path('../results/generation_probe.json')
# PHASE = 'decode'   # 'decode' or 'prefill'
SAVE_FIG = False
OUT_DIR = Path('../results')
TRIM_EDGE_LAYERS = False
# -----------------

data = json.loads(RESULT_PATH.read_text(encoding='utf-8'))
runs = data['runs'] if 'runs' in data else [data]
print(f'loaded runs: {len(runs)} from {RESULT_PATH}')


def pick_steps(run, phase='decode'):
    timeline = run.get('timeline') or []
    if timeline:
        picked = [s for s in timeline if s.get('phase') == phase]
        return picked if picked else timeline
    if phase == 'prefill':
        prefill = run.get('prefill_steps') or []
        if prefill:
            return prefill
    return run.get('steps', [])


def collect_layerwise_series_from_layer_metrics(steps, value_key):
    by_layer = {}
    for s in steps:
        for m in (s.get('layer_metrics') or []):
            l = m.get('layer')
            v = m.get(value_key)
            if l is None or v is None:
                continue
            by_layer.setdefault(int(l), []).append(float(v))

    layers = sorted(by_layer)
    if TRIM_EDGE_LAYERS and len(layers) > 2:
        layers = layers[1:-1]

    return {l: by_layer[l] for l in layers}


def collect_layerwise_series_from_sublayer(steps, metric_key, value_key):
    by_layer = {}
    for s in steps:
        for item in (s.get('sublayer_metrics') or []):
            l = item.get('layer')
            v = (item.get(metric_key) or {}).get(value_key)
            if l is None or v is None:
                continue
            by_layer.setdefault(int(l), []).append(float(v))

    layers = sorted(by_layer)
    if TRIM_EDGE_LAYERS and len(layers) > 2:
        layers = layers[1:-1]

    return {l: by_layer[l] for l in layers}


def build_ratio_stats(z_series_by_layer, dz_series_by_layer):
    layers = sorted(set(z_series_by_layer.keys()) & set(dz_series_by_layer.keys()))
    if not layers:
        return [], [], [], [], []

    mean_r, std_r, lo_r, hi_r = [], [], [], []
    kept_layers = []

    for l in layers:
        z_vals = z_series_by_layer[l]
        dz_vals = dz_series_by_layer[l]
        n = min(len(z_vals), len(dz_vals))
        if n == 0:
            continue

        ratios = np.asarray([z_vals[i] / (dz_vals[i] + 1e-12) for i in range(n)], dtype=float)
        m = float(np.mean(ratios))
        s = float(np.std(ratios))

        kept_layers.append(l)
        mean_r.append(m)
        std_r.append(s)
        lo_r.append(m - s)
        hi_r.append(m + s)

    if not kept_layers:
        return [], [], [], [], []

    return kept_layers, mean_r, std_r, lo_r, hi_r


series = [('block', 'BLOCK', 'block_update'), ('attn', 'ATTN', 'attn_update'), ('mlp', 'MLP', 'mlp_update')]

for prompt_idx, run in enumerate(runs):
    steps = pick_steps(run, phase=PHASE)
    if not steps:
        print(f'prompt {prompt_idx}: no steps ({PHASE}), skipped')
        continue

    has_sublayer = any(('sublayer_metrics' in s and s['sublayer_metrics']) for s in steps)
    has_layer = any(('layer_metrics' in s and s['layer_metrics']) for s in steps)
    print(f'prompt {prompt_idx}: steps={len(steps)}, phase={PHASE}, has_sublayer={has_sublayer}, has_layer={has_layer}')

    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=False)

    for ax, (name, title, metric_key) in zip(axes, series):
        used_layer_z_fallback = False
        used_layer_dz_fallback = False

        if has_sublayer:
            if name == 'block':
                z_series = collect_layerwise_series_from_sublayer(steps, 'block_update', 'z_post_norm')
                dz_series = collect_layerwise_series_from_sublayer(steps, 'block_update', 'dz_norm')
                if has_layer and not z_series:
                    z_series = collect_layerwise_series_from_layer_metrics(steps, 'z_post_norm')
                    used_layer_z_fallback = True
                if has_layer and not dz_series:
                    dz_series = collect_layerwise_series_from_layer_metrics(steps, 'dz_norm')
                    used_layer_dz_fallback = True
            else:
                z_series = collect_layerwise_series_from_sublayer(steps, metric_key, 'z_post_norm')
                dz_series = collect_layerwise_series_from_sublayer(steps, metric_key, 'dz_norm')
        else:
            z_series = collect_layerwise_series_from_layer_metrics(steps, 'z_post_norm')
            dz_series = collect_layerwise_series_from_layer_metrics(steps, 'dz_norm')

        x_r, mean_r, std_r, lo_r, hi_r = build_ratio_stats(z_series, dz_series)

        if x_r:
            ax.plot(x_r, mean_r, color='tab:green', lw=1.9, label='z_post_norm/dz_norm mean')
            ax.fill_between(x_r, lo_r, hi_r, color='tab:green', alpha=0.14, label='mean ± std')

            curve_mean = float(np.mean(mean_r))
            curve_var = float(np.var(mean_r))
            print(f'[prompt {prompt_idx}] {title}: curve_mean={curve_mean:.6f}, curve_var={curve_var:.6f}')
        else:
            ax.text(0.5, 0.5, 'No ratio data', ha='center', va='center', transform=ax.transAxes)

        ax.set_title(f'{title} ratio (mean ± std over steps)')
        ax.set_xlabel('Layer')
        ax.set_ylabel('z_post_norm / dz_norm', color='tab:green')
        ax.grid(alpha=0.25)
        lines, labels = ax.get_legend_handles_labels()
        if lines:
            ax.legend(loc='best')

        if used_layer_z_fallback:
            ax.text(0.01, 0.02, 'z from layer_metrics', transform=ax.transAxes, fontsize=8, color='tab:blue')
        if used_layer_dz_fallback:
            ax.text(0.01, 0.08, 'dz from layer_metrics', transform=ax.transAxes, fontsize=8, color='tab:orange')

    fig.suptitle(f'Prompt {prompt_idx}: z_post_norm/dz_norm (mean ± std, phase={PHASE})')
    fig.tight_layout()

    if SAVE_FIG:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = OUT_DIR / f'prompt{prompt_idx}_{PHASE}_layerwise_z_over_dz_mean_std.png'
        fig.savefig(out_path, dpi=150)
        print(f'saved: {out_path}')

    plt.show()
    break


In [ ]:
# NEW BLOCK: compare two JSONs (supports different model depths)
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# ---- config ----
RESULT_PATH_A = Path('results/generation_probe.json')
RESULT_PATH_B = Path('results/generation_probe.json')
LABEL_A = 'Model A'
LABEL_B = 'Model B'
PHASE = 'decode'       # 'decode' or 'prefill'
PART = 'attn'          # 'block' | 'attn' | 'mlp'
SAVE_FIG = False
OUT_DIR = Path('../results')
# -----------------


def load_runs(path: Path):
    data = json.loads(path.read_text(encoding='utf-8'))
    return data['runs'] if 'runs' in data else [data]


def pick_steps(run, phase='decode'):
    timeline = run.get('timeline') or []
    if timeline:
        picked = [s for s in timeline if s.get('phase') == phase]
        return picked if picked else timeline
    if phase == 'prefill':
        prefill = run.get('prefill_steps') or []
        if prefill:
            return prefill
    return run.get('steps', [])



def _compute_ratio_from_item(d):
    if not d:
        return None
    z = d.get('z_post_norm')
    dz = d.get('dz_norm')
    if z is None or dz is None:
        return None
    return float(z) / (float(dz) + 1e-12)

def collect_layerwise_series(steps, part='attn', value_key='z_over_dz_ratio'):
    by_layer = {}
    if part == 'block':
        # Prefer sublayer block_update, fallback to layer_metrics
        has_sublayer = any(('sublayer_metrics' in s and s['sublayer_metrics']) for s in steps)
        if has_sublayer:
            for s in steps:
                for item in (s.get('sublayer_metrics') or []):
                    l = item.get('layer')
                    v = _compute_ratio_from_item(item.get('block_update')) if value_key == 'z_over_dz_ratio' else (item.get('block_update') or {}).get(value_key)
                    if l is None or v is None:
                        continue
                    by_layer.setdefault(int(l), []).append(float(v))
        if not by_layer:
            for s in steps:
                for m in (s.get('layer_metrics') or []):
                    l = m.get('layer')
                    v = _compute_ratio_from_item(m) if value_key == 'z_over_dz_ratio' else m.get(value_key)
                    if l is None or v is None:
                        continue
                    by_layer.setdefault(int(l), []).append(float(v))
    else:
        sub_key = 'attn_update' if part == 'attn' else 'mlp_update'
        for s in steps:
            for item in (s.get('sublayer_metrics') or []):
                l = item.get('layer')
                v = _compute_ratio_from_item(item.get(sub_key)) if value_key == 'z_over_dz_ratio' else (item.get(sub_key) or {}).get(value_key)
                if l is None or v is None:
                    continue
                by_layer.setdefault(int(l), []).append(float(v))

    layers = sorted(by_layer)
    if not layers:
        return [], [], [], []

    means = [float(np.mean(by_layer[l])) for l in layers]
    mins = [float(np.min(by_layer[l])) for l in layers]
    maxs = [float(np.max(by_layer[l])) for l in layers]
    return layers, means, mins, maxs


def aggregate_prompt_layer_curves(runs, phase='decode', part='attn', value_key='z_over_dz_ratio'):
    # prompt-aggregated layer curve: for each layer, aggregate over prompts
    all_layers = set()
    per_prompt = []

    for run in runs:
        steps = pick_steps(run, phase=phase)
        x, m, _, _ = collect_layerwise_series(steps, part=part, value_key=value_key)
        if x:
            all_layers.update(x)
            per_prompt.append((x, m))

    if not all_layers:
        return [], [], [], []

    layers = sorted(all_layers)
    vals = {l: [] for l in layers}
    for x, m in per_prompt:
        d = {lx: mv for lx, mv in zip(x, m)}
        for l in layers:
            if l in d:
                vals[l].append(d[l])

    mean = [float(np.mean(vals[l])) if vals[l] else np.nan for l in layers]
    lo = [float(np.min(vals[l])) if vals[l] else np.nan for l in layers]
    hi = [float(np.max(vals[l])) if vals[l] else np.nan for l in layers]
    return layers, mean, lo, hi


def to_relative_depth(layers):
    if not layers:
        return []
    lmax = max(layers)
    if lmax <= 0:
        return [0.0 for _ in layers]
    return [l / lmax for l in layers]


def summarize_curve(y):
    y = np.asarray([v for v in y if np.isfinite(v)], dtype=float)
    if y.size == 0:
        return None, None
    return float(np.mean(y)), float(np.var(y))


runs_a = load_runs(RESULT_PATH_A)
runs_b = load_runs(RESULT_PATH_B)

x_a, y_a, lo_a, hi_a = aggregate_prompt_layer_curves(runs_a, phase=PHASE, part=PART, value_key='z_over_dz_ratio')
x_b, y_b, lo_b, hi_b = aggregate_prompt_layer_curves(runs_b, phase=PHASE, part=PART, value_key='z_over_dz_ratio')

print(f'{LABEL_A}: runs={len(runs_a)}, layers={len(x_a)}')
print(f'{LABEL_B}: runs={len(runs_b)}, layers={len(x_b)}')

m_a, v_a = summarize_curve(y_a)
m_b, v_b = summarize_curve(y_b)
print(f'{LABEL_A} curve mean={m_a}, var={v_a}')
print(f'{LABEL_B} curve mean={m_b}, var={v_b}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# left: raw layer axis
ax = axes[0]
if x_a:
    ax.plot(x_a, y_a, color='tab:blue', lw=1.8, label=LABEL_A)
    ax.fill_between(x_a, lo_a, hi_a, color='tab:blue', alpha=0.12)
if x_b:
    ax.plot(x_b, y_b, color='tab:orange', lw=1.8, label=LABEL_B)
    ax.fill_between(x_b, lo_b, hi_b, color='tab:orange', alpha=0.12)
ax.set_title('Raw Layer Index')
ax.set_xlabel('Layer')
ax.set_ylabel('z_post_norm / dz_norm')
ax.grid(alpha=0.25)
ax.legend(loc='best')

# right: normalized depth axis (supports different depths)
ax2 = axes[1]
rx_a = to_relative_depth(x_a)
rx_b = to_relative_depth(x_b)
if rx_a:
    ax2.plot(rx_a, y_a, color='tab:blue', lw=1.8, label=LABEL_A)
    ax2.fill_between(rx_a, lo_a, hi_a, color='tab:blue', alpha=0.12)
if rx_b:
    ax2.plot(rx_b, y_b, color='tab:orange', lw=1.8, label=LABEL_B)
    ax2.fill_between(rx_b, lo_b, hi_b, color='tab:orange', alpha=0.12)
ax2.set_title('Relative Depth (layer / last_layer)')
ax2.set_xlabel('Relative depth [0, 1]')
ax2.set_ylabel('z_post_norm / dz_norm')
ax2.grid(alpha=0.25)
ax2.legend(loc='best')

fig.suptitle(f'Compare {PART.upper()} | phase={PHASE}')
fig.tight_layout()

if SAVE_FIG:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out = OUT_DIR / f'compare_two_json_{PART}_{PHASE}.png'
    fig.savefig(out, dpi=160)
    print('saved:', out)

plt.show()


In [ ]:
# NEW BLOCK: z_post_norm / dz_norm with variance band, two-model overlay (per prompt, layer-wise)
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

# ---- config ----
RESULT_PATH_A = Path('results/generation_probe.json')
RESULT_PATH_B = Path('results/generation_probe.json')
LABEL_A = 'Model A'
LABEL_B = 'Model B'
PHASE = 'decode'   # 'decode' or 'prefill'
SAVE_FIG = False
OUT_DIR = Path('../results')
TRIM_EDGE_LAYERS = False
# -----------------

data_a = json.loads(RESULT_PATH_A.read_text(encoding='utf-8'))
runs_a = data_a['runs'] if 'runs' in data_a else [data_a]

data_b = json.loads(RESULT_PATH_B.read_text(encoding='utf-8'))
runs_b = data_b['runs'] if 'runs' in data_b else [data_b]

print(f'loaded A runs: {len(runs_a)} from {RESULT_PATH_A}')
print(f'loaded B runs: {len(runs_b)} from {RESULT_PATH_B}')


def pick_steps(run, phase='decode'):
    timeline = run.get('timeline') or []
    if timeline:
        picked = [s for s in timeline if s.get('phase') == phase]
        return picked if picked else timeline
    if phase == 'prefill':
        prefill = run.get('prefill_steps') or []
        if prefill:
            return prefill
    return run.get('steps', [])


def collect_layerwise_series_from_layer_metrics(steps, value_key):
    by_layer = {}
    for s in steps:
        for m in (s.get('layer_metrics') or []):
            l = m.get('layer')
            v = m.get(value_key)
            if l is None or v is None:
                continue
            by_layer.setdefault(int(l), []).append(float(v))

    layers = sorted(by_layer)
    if TRIM_EDGE_LAYERS and len(layers) > 2:
        layers = layers[1:-1]

    return {l: by_layer[l] for l in layers}


def collect_layerwise_series_from_sublayer(steps, metric_key, value_key):
    by_layer = {}
    for s in steps:
        for item in (s.get('sublayer_metrics') or []):
            l = item.get('layer')
            v = (item.get(metric_key) or {}).get(value_key)
            if l is None or v is None:
                continue
            by_layer.setdefault(int(l), []).append(float(v))

    layers = sorted(by_layer)
    if TRIM_EDGE_LAYERS and len(layers) > 2:
        layers = layers[1:-1]

    return {l: by_layer[l] for l in layers}


def build_ratio_stats(z_series_by_layer, dz_series_by_layer):
    layers = sorted(set(z_series_by_layer.keys()) & set(dz_series_by_layer.keys()))
    if not layers:
        return [], [], [], [], []

    mean_r, std_r, lo_r, hi_r = [], [], [], []
    kept_layers = []

    for l in layers:
        z_vals = z_series_by_layer[l]
        dz_vals = dz_series_by_layer[l]
        n = min(len(z_vals), len(dz_vals))
        if n == 0:
            continue

        ratios = np.asarray([z_vals[i] / (dz_vals[i] + 1e-12) for i in range(n)], dtype=float)
        m = float(np.mean(ratios))
        s = float(np.std(ratios))

        kept_layers.append(l)
        mean_r.append(m)
        std_r.append(s)
        lo_r.append(m - s)
        hi_r.append(m + s)

    if not kept_layers:
        return [], [], [], [], []

    return kept_layers, mean_r, std_r, lo_r, hi_r


def get_ratio_for_part(steps, part_name, metric_key):
    has_sublayer = any(('sublayer_metrics' in s and s['sublayer_metrics']) for s in steps)
    has_layer = any(('layer_metrics' in s and s['layer_metrics']) for s in steps)

    if has_sublayer:
        if part_name == 'block':
            z_series = collect_layerwise_series_from_sublayer(steps, 'block_update', 'z_post_norm')
            dz_series = collect_layerwise_series_from_sublayer(steps, 'block_update', 'dz_norm')
            if has_layer and not z_series:
                z_series = collect_layerwise_series_from_layer_metrics(steps, 'z_post_norm')
            if has_layer and not dz_series:
                dz_series = collect_layerwise_series_from_layer_metrics(steps, 'dz_norm')
        else:
            z_series = collect_layerwise_series_from_sublayer(steps, metric_key, 'z_post_norm')
            dz_series = collect_layerwise_series_from_sublayer(steps, metric_key, 'dz_norm')
    else:
        z_series = collect_layerwise_series_from_layer_metrics(steps, 'z_post_norm')
        dz_series = collect_layerwise_series_from_layer_metrics(steps, 'dz_norm')

    return build_ratio_stats(z_series, dz_series)


series = [('block', 'BLOCK', 'block_update'), ('attn', 'ATTN', 'attn_update'), ('mlp', 'MLP', 'mlp_update')]

n_prompts = min(len(runs_a), len(runs_b))
for prompt_idx in range(n_prompts):
    run_a = runs_a[prompt_idx]
    run_b = runs_b[prompt_idx]

    steps_a = pick_steps(run_a, phase=PHASE)
    steps_b = pick_steps(run_b, phase=PHASE)

    if not steps_a or not steps_b:
        print(f'prompt {prompt_idx}: no steps in one side (phase={PHASE}), skipped')
        continue

    print(f'prompt {prompt_idx}: A_steps={len(steps_a)}, B_steps={len(steps_b)}, phase={PHASE}')

    fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharex=False)

    for ax, (name, title, metric_key) in zip(axes, series):
        x_a, mean_a, std_a, lo_a, hi_a = get_ratio_for_part(steps_a, name, metric_key)
        x_b, mean_b, std_b, lo_b, hi_b = get_ratio_for_part(steps_b, name, metric_key)

        if x_a:
            ax.plot(x_a, mean_a, color='tab:green', lw=1.9, label=f'{LABEL_A} mean')
            ax.fill_between(x_a, lo_a, hi_a, color='tab:green', alpha=0.14, label=f'{LABEL_A} mean±std')
            print(f'[prompt {prompt_idx}] {title} {LABEL_A}: curve_mean={float(np.mean(mean_a)):.6f}, curve_var={float(np.var(mean_a)):.6f}')

        if x_b:
            ax.plot(x_b, mean_b, color='tab:purple', lw=1.9, label=f'{LABEL_B} mean')
            ax.fill_between(x_b, lo_b, hi_b, color='tab:purple', alpha=0.12, label=f'{LABEL_B} mean±std')
            print(f'[prompt {prompt_idx}] {title} {LABEL_B}: curve_mean={float(np.mean(mean_b)):.6f}, curve_var={float(np.var(mean_b)):.6f}')

        if not x_a and not x_b:
            ax.text(0.5, 0.5, 'No ratio data', ha='center', va='center', transform=ax.transAxes)

        ax.set_title(f'{title} ratio (mean ± std over steps)')
        ax.set_xlabel('Layer')
        ax.set_ylabel('z_post_norm / dz_norm')
        ax.grid(alpha=0.25)

        lines, labels = ax.get_legend_handles_labels()
        if lines:
            ax.legend(loc='best', fontsize=8)

    fig.suptitle(f'Prompt {prompt_idx}: z_post_norm/dz_norm overlay ({LABEL_A} vs {LABEL_B}, phase={PHASE})')
    fig.tight_layout()

    if SAVE_FIG:
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = OUT_DIR / f'prompt{prompt_idx}_{PHASE}_layerwise_z_over_dz_overlay.png'
        fig.savefig(out_path, dpi=150)
        print(f'saved: {out_path}')

    plt.show()
    break
